# UABAF User Guide — Comprehensive Example Notebook
**Package version:** uabaf 0.2.0  
**Author:** Balgah Sounders Junior  
**Institution:** University of Buea, Department of Computer Engineering

---

This notebook is a complete user guide for the UABAF Python package. It walks through every way the package can be used — from the quickest one-call audit to calling each module individually for full control. It also shows how to apply custom fairness thresholds for stricter or more permissive domains.

### What this notebook covers

| Section | What you will learn |
|---------|--------------------|
| 1. Installation | How to install uabaf |
| 2. Dataset setup | Loading and preparing COMPAS for the audit |
| 3. Module 1 — Profiler | Running Stage 1 dataset readiness checks |
| 4. Module 2 — Metrics | Computing the four fairness metrics directly |
| 5. Module 3 — Bootstrap | Computing BCa confidence intervals per metric |
| 6. Module 4 — Verdict | Assigning uncertainty-aware verdicts |
| 7. Module 5 — Visualise | Plotting CI intervals and verdict summary grids |
| 8. Module 6 — AuditReport | Running the full pipeline in one call |
| 9. Custom thresholds | Using stricter or looser fairness standards |
| 10. Saving results | Exporting to CSV and PNG |

---

### The four verdict categories

Every metric result in UABAF receives one of four verdicts, determined by two questions:
1. Does the point estimate pass or fail the fairness threshold?
2. Is the confidence interval width at or below 0.15?

| Verdict | Meaning |
|---------|--------|
| PASS with High Confidence | Metric passes AND interval is narrow. Reliable result. |
| PASS with Low Confidence | Metric passes BUT interval is wide. Collect more data. |
| FAIL with High Confidence | Metric fails AND interval is narrow. Reliable failure. |
| FAIL with Low Confidence | Metric fails BUT interval is wide. Inconclusive. Investigate. |

---
## Section 1: Installation

In [ ]:
# Install uabaf from PyPI
# Run this cell once. If uabaf is already installed, this will upgrade to the latest version.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'uabaf', '--quiet'])
print('uabaf installed.')

In [ ]:
# Confirm the version
import uabaf
print(f'uabaf version: {uabaf.__version__}')
# Expected: 0.2.0

---
## Section 2: Dataset Setup

We use the COMPAS recidivism dataset throughout this guide.  
- **Task:** predict whether a defendant will reoffend within two years  
- **Sensitive attribute:** race (0 = Caucasian, 1 = African-American)  
- **Target:** two_year_recid (0 = did not reoffend, 1 = reoffended)

The dataset is subsampled to 600 records to demonstrate UABAF in a small-dataset setting.

In [ ]:
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedShuffleSplit
warnings.filterwarnings('ignore')
np.random.seed(42)

# Load COMPAS from ProPublica GitHub
url = ('https://raw.githubusercontent.com/propublica/compas-analysis/'
       'master/compas-scores-two-years.csv')
df_raw = pd.read_csv(url)

# Select and clean columns
keep = ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count',
        'priors_count', 'c_charge_degree', 'sex', 'race', 'two_year_recid']
df = df_raw[keep].dropna()
df = df[df['race'].isin(['African-American', 'Caucasian'])].copy()

# Encode features
df['race_bin']   = (df['race'] == 'African-American').astype(int)
df['sex_bin']    = (df['sex'] == 'Male').astype(int)
df['charge_bin'] = (df['c_charge_degree'] == 'F').astype(int)
df = df.drop(columns=['race', 'sex', 'c_charge_degree'])

# Stratified subsample to 600 records
sss = StratifiedShuffleSplit(n_splits=1, test_size=600/len(df), random_state=42)
for _, idx in sss.split(df, df['two_year_recid']):
    df = df.iloc[idx].reset_index(drop=True)

sensitive_col = 'race_bin'
target_col    = 'two_year_recid'
feature_cols  = [c for c in df.columns if c not in [sensitive_col, target_col]]

print(f'Dataset shape  : {df.shape}')
print(f'Group counts   : {df[sensitive_col].value_counts().to_dict()}')
print(f'Class balance  : {df[target_col].value_counts().to_dict()}')
print(f'Features       : {feature_cols}')

In [ ]:
# 70/30 stratified train/test split
strat = df[target_col].astype(str) + '_' + df[sensitive_col].astype(str)
sss2  = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
for train_idx, test_idx in sss2.split(df[feature_cols], strat):
    X_train = df.iloc[train_idx][feature_cols].values
    X_test  = df.iloc[test_idx][feature_cols].values
    y_train = df.iloc[train_idx][target_col].values
    y_test  = df.iloc[test_idx][target_col].values
    s_train = df.iloc[train_idx][sensitive_col].values
    s_test  = df.iloc[test_idx][sensitive_col].values

# Train a Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'Train size : {len(y_train)}')
print(f'Test size  : {len(y_test)}')
print(f'Test group counts — 0: {(s_test==0).sum()}, 1: {(s_test==1).sum()}')

---
## Section 3: Module 1 — Profiler

The profiler runs Stage 1 of the UABAF pipeline. It checks the dataset itself before any metric is computed. Call it directly when you want to inspect data readiness without running the full audit.

**Four checks performed:**
1. Subgroup size — are both groups large enough for reliable intervals?
2. Class imbalance — is the target label heavily skewed within either group?
3. Proxy correlation — does any feature encode the sensitive attribute indirectly?
4. Missing data — are there columns with too many missing values?

In [ ]:
from uabaf.profiler import profile_dataset

# Run Stage 1 on the full COMPAS dataset (before splitting)
profile = profile_dataset(
    df           = df,
    features     = feature_cols,
    sensitive    = sensitive_col,
    target       = target_col,
    dataset_name = 'COMPAS (n=600)'
)

# Print the formatted report
profile.print_report()

In [ ]:
# Access the findings programmatically as a DataFrame
findings_df = profile.to_dataframe()
print(f'Overall passed: {profile.passed}')
print(f'Summary       : {profile.summary}')
print()
print(findings_df[['check', 'severity', 'feature', 'value', 'message']])

In [ ]:
# The profiler thresholds can also be adjusted
# Example: stricter proxy threshold of 0.30 instead of the default 0.40
profile_strict = profile_dataset(
    df               = df,
    features         = feature_cols,
    sensitive        = sensitive_col,
    target           = target_col,
    dataset_name     = 'COMPAS — Strict Proxy Check',
    proxy_threshold  = 0.30,   # flag anything above 0.30
    min_warning      = 80,     # warn if any group is below 80
)
profile_strict.print_report()

---
## Section 4: Module 2 — Metrics

The metrics module computes the four fairness metrics directly on any set of predictions. Use this when you only need the point estimates — for example to compare two models quickly before running the full bootstrap audit.

| Metric | Measures | Fair if |
|--------|----------|---------|
| DPD | Difference in positive prediction rates | \|DPD\| <= 0.10 |
| EOD | Largest of TPR diff and FPR diff | EOD <= 0.10 |
| EOP | Difference in True Positive Rates only | EOP <= 0.10 |
| DI  | Ratio of positive prediction rates | 0.80 <= DI <= 1.20 |

In [ ]:
from uabaf.metrics import (
    fairness_metrics,
    demographic_parity_diff,
    equalized_odds_diff,
    equal_opportunity_diff,
    disparate_impact_ratio,
)

# Option A: compute all four metrics at once (recommended)
metrics = fairness_metrics(y_test, y_pred, s_test)

print('All four metrics at once:')
print(f"  DPD : {metrics['dpd']:.4f}")
print(f"  EOD : {metrics['eod']:.4f}")
print(f"  EOP : {metrics['eop']:.4f}")
print(f"  DI  : {metrics['di']:.4f}")

In [ ]:
# Option B: compute individual metrics separately
dpd = demographic_parity_diff(y_test, y_pred, s_test)
eod = equalized_odds_diff(y_test, y_pred, s_test)
eop = equal_opportunity_diff(y_test, y_pred, s_test)
di  = disparate_impact_ratio(y_test, y_pred, s_test)

print('Individual metric calls:')
print(f'  DPD : {dpd:.4f}  (threshold: |DPD| <= 0.10  ->  {"PASS" if abs(dpd) <= 0.10 else "FAIL"})')
print(f'  EOD : {eod:.4f}  (threshold: EOD  <= 0.10   ->  {"PASS" if eod <= 0.10 else "FAIL"})')
print(f'  EOP : {eop:.4f}  (threshold: EOP  <= 0.10   ->  {"PASS" if eop <= 0.10 else "FAIL"})')
print(f'  DI  : {di:.4f}  (threshold: 0.80 <= DI <= 1.20  ->  {"PASS" if 0.80 <= di <= 1.20 else "FAIL"})')

In [ ]:
# Compare two models side by side using just the metrics module
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)

metrics_lr = fairness_metrics(y_test, y_pred,    s_test)
metrics_rf = fairness_metrics(y_test, y_pred_rf, s_test)

print(f'{"Metric":<6}  {"Logistic Regression":>22}  {"Random Forest":>15}')
print('-' * 48)
for key in ['dpd', 'eod', 'eop', 'di']:
    print(f'{key.upper():<6}  {metrics_lr[key]:>22.4f}  {metrics_rf[key]:>15.4f}')

---
## Section 5: Module 3 — Bootstrap

The bootstrap module computes BCa confidence intervals for any fairness metric. This is Stage 2 of the pipeline. Each result includes the point estimate, the CI lower and upper bounds, and the CI width.

A CI width <= 0.15 means high confidence. A width > 0.15 means low confidence.

Two functions are available:
- `bca_ci` — one metric at a time
- `bca_ci_all_metrics` — all four metrics in a single efficient pass (recommended)

In [ ]:
from uabaf.bootstrap import bca_ci, bca_ci_all_metrics

# Option A: single metric CI
# Use this when you only need one specific metric
dpd_result = bca_ci(
    y_true       = y_test,
    y_pred       = y_pred,
    sensitive    = s_test,
    metric_key   = 'dpd',
    B            = 2000,    # number of bootstrap resamples
    alpha        = 0.05,    # significance level -> 95% CI
    random_state = 42,
)

print('Single metric CI result:')
print(f"  Metric      : {dpd_result['metric'].upper()}")
print(f"  Point est.  : {dpd_result['point']:.4f}")
print(f"  CI lower    : {dpd_result['ci_lo']:.4f}")
print(f"  CI upper    : {dpd_result['ci_hi']:.4f}")
print(f"  CI width    : {dpd_result['width']:.4f}")
print(f"  Bootstrap n : {dpd_result['n']}")

In [ ]:
# Option B: all four metrics in one efficient pass
# This runs the bootstrap loop once and computes all four metrics per resample
# much faster than calling bca_ci four times separately
ci_results = bca_ci_all_metrics(
    y_true       = y_test,
    y_pred       = y_pred,
    sensitive    = s_test,
    B            = 2000,
    alpha        = 0.05,
    random_state = 42,
)

print('All four metrics — BCa 95% Confidence Intervals:')
print(f"{'Metric':<6}  {'Point':>8}  {'CI Lower':>10}  {'CI Upper':>10}  {'Width':>8}")
print('-' * 52)
for key, res in ci_results.items():
    print(f"{key.upper():<6}  {res['point']:>8.4f}  {res['ci_lo']:>10.4f}  {res['ci_hi']:>10.4f}  {res['width']:>8.4f}")

In [ ]:
# You can also change the confidence level
# Example: 90% CI instead of 95%
ci_90 = bca_ci_all_metrics(
    y_true       = y_test,
    y_pred       = y_pred,
    sensitive    = s_test,
    B            = 2000,
    alpha        = 0.10,    # alpha=0.10 gives 90% CI
    random_state = 42,
)

print('Comparison: 95% CI vs 90% CI for DPD')
print(f"  95% CI: [{ci_results['dpd']['ci_lo']:.4f}, {ci_results['dpd']['ci_hi']:.4f}]  "
      f"width={ci_results['dpd']['width']:.4f}")
print(f"  90% CI: [{ci_90['dpd']['ci_lo']:.4f}, {ci_90['dpd']['ci_hi']:.4f}]  "
      f"width={ci_90['dpd']['width']:.4f}")
print()
print('The 90% CI is narrower because it requires less certainty.')

---
## Section 6: Module 4 — Verdict

The verdict module takes the CI results from the bootstrap module and assigns one of four verdicts to each metric. This is Stage 3 of the pipeline.

Two functions are available:
- `assign_verdict` — one metric at a time
- `assign_all_verdicts` — all four from a ci_results dict

In [ ]:
from uabaf.verdict import assign_verdict, assign_all_verdicts, THRESHOLDS, CI_WIDTH_THRESHOLD

# Show the default thresholds built into the package
print('Default UABAF fairness thresholds:')
for metric, (kind, value) in THRESHOLDS.items():
    if kind == 'diff':
        print(f'  {metric.upper():<5}: |value| <= {value}  (difference threshold)')
    else:
        print(f'  {metric.upper():<5}: value in {value}  (ratio threshold)')

print(f'\nDefault CI width threshold: {CI_WIDTH_THRESHOLD}')
print('  CI width <= 0.15 -> High Confidence')
print('  CI width >  0.15 -> Low Confidence')

In [ ]:
# Option A: assign verdict for one metric
dpd_verdict = assign_verdict(
    metric_key = 'dpd',
    point      = ci_results['dpd']['point'],
    ci_lo      = ci_results['dpd']['ci_lo'],
    ci_hi      = ci_results['dpd']['ci_hi'],
)

print('Single metric verdict:')
print(f"  Metric     : {dpd_verdict['label']}")
print(f"  Point est. : {dpd_verdict['point']:.4f}"  if 'point' not in dpd_verdict else f"  Point est. : {ci_results['dpd']['point']:.4f}")
print(f"  Verdict    : {dpd_verdict['verdict']}")
print(f"  Pass/Fail  : {dpd_verdict['pass_fail']}")
print(f"  Confidence : {dpd_verdict['confidence']}")
print(f"  Breaches   : {dpd_verdict['breaches']}")
print(f"  CI Width   : {dpd_verdict['width']:.4f}")

In [ ]:
# Option B: assign verdicts for all four metrics at once
verdicts = assign_all_verdicts(ci_results)

print(f'{"Metric":<30} {"Point":>7}  {"CI Width":>9}  Verdict')
print('-' * 75)
for key, v in verdicts.items():
    print(f"{v['label']:<30} {v['point']:>7.4f}  {v['width']:>9.4f}  {v['verdict']}")

---
## Section 7: Module 5 — Visualise

The visualise module produces two types of plots:

1. `plot_ci_intervals` — horizontal bar chart showing BCa CIs for all four metrics, coloured by verdict
2. `plot_verdict_summary` — a grid comparing verdicts across multiple models or datasets

In [ ]:
from uabaf.visualise import plot_ci_intervals, plot_verdict_summary

# Plot 1: CI interval chart for one model
fig = plot_ci_intervals(
    verdict_results = verdicts,
    title           = 'COMPAS — Logistic Regression — BCa 95% CIs'
)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: verdict summary grid comparing two models
# First get CI results and verdicts for the Random Forest model
ci_results_rf = bca_ci_all_metrics(
    y_true       = y_test,
    y_pred       = y_pred_rf,
    sensitive    = s_test,
    B            = 2000,
    random_state = 42,
)
verdicts_rf = assign_all_verdicts(ci_results_rf)

# Build the comparison dict
# Keys become row labels in the grid
summary_data = {
    'Logistic Regression': verdicts,
    'Random Forest'      : verdicts_rf,
}

fig2 = plot_verdict_summary(
    results_dict = summary_data,
    title        = 'COMPAS — Model Comparison'
)
plt.tight_layout()
plt.show()

In [ ]:
# Save a plot to file
fig = plot_ci_intervals(verdicts, title='COMPAS LR — Saved Plot')
fig.savefig('compas_lr_ci_plot.png', dpi=150, bbox_inches='tight')
print('Plot saved to compas_lr_ci_plot.png')
plt.show()

---
## Section 8: Module 6 — AuditReport

The AuditReport class is the simplest way to use UABAF. It runs all three stages automatically and exposes the results through a clean interface. Under the hood it calls `profile_dataset`, `bca_ci_all_metrics`, and `assign_all_verdicts` for you.

Use this when you want a complete audit without managing each stage separately.

In [ ]:
from uabaf import AuditReport

# Full audit in one call
report = AuditReport(
    model        = model,           # fitted sklearn model
    X_test       = X_test,          # test features
    y_test       = y_test,          # true labels
    sensitive    = s_test,          # sensitive attribute vector
    B            = 2000,            # bootstrap resamples
    alpha        = 0.05,            # significance level
    random_state = 42,
    model_name   = 'Logistic Regression',
    dataset_name = 'COMPAS (n=600)',
)

# Print the Stage 3 verdict table
report.summary(show_stage1=False)

In [ ]:
# Include Stage 1 in the summary
# This requires passing the original DataFrame to the AuditReport
report_full = AuditReport(
    model        = model,
    X_test       = X_test,
    y_test       = y_test,
    sensitive    = s_test,
    B            = 2000,
    random_state = 42,
    model_name   = 'Logistic Regression',
    dataset_name = 'COMPAS (n=600)',
    df           = df,              # pass the full dataframe for Stage 1
    features     = feature_cols,
    target       = target_col,
)

report_full.summary(show_stage1=True)

In [ ]:
# Generate the CI plot from the report
fig = report.plot()
plt.show()

In [ ]:
# Export results to a DataFrame
df_results = report.to_dataframe()
print(df_results[['metric', 'label', 'point', 'ci_lo', 'ci_hi', 'width', 'verdict']])

In [ ]:
# Check the overall pass/fail and repr
print(f'Overall passed: {report.passed}')
print(f'Repr          : {repr(report)}')

---
## Section 9: Custom Thresholds

The default fairness thresholds in UABAF are:
- Difference metrics (DPD, EOD, EOP): |value| <= 0.10
- Disparate Impact: 0.80 <= value <= 1.20
- CI width: <= 0.15 for high confidence

These can be overridden at any level — in individual metric verdict calls, in `assign_all_verdicts`, or directly in `AuditReport`. This is useful when your domain requires stricter or more permissive standards.

In [ ]:
# Define stricter thresholds
# Useful for high-stakes applications such as healthcare or criminal justice
strict_thresholds = {
    'dpd': ('diff',  0.05),          # max 5 percentage points difference
    'eod': ('diff',  0.05),
    'eop': ('diff',  0.05),
    'di' : ('ratio', (0.90, 1.10)),  # must be within 90%-110%
}

strict_verdicts = assign_all_verdicts(
    bootstrap_results  = ci_results,
    thresholds         = strict_thresholds,
    ci_width_threshold = 0.10,   # stricter CI width threshold too
)

print('Strict threshold verdicts (threshold = 0.05, CI width = 0.10):')
print(f'{"Metric":<30} {"Point":>7}  Verdict')
print('-' * 60)
for key, v in strict_verdicts.items():
    print(f"{v['label']:<30} {v['point']:>7.4f}  {v['verdict']}")

In [ ]:
# Define lenient thresholds
# Useful for lower-stakes applications or exploratory work
lenient_thresholds = {
    'dpd': ('diff',  0.20),
    'eod': ('diff',  0.20),
    'eop': ('diff',  0.20),
    'di' : ('ratio', (0.70, 1.30)),
}

lenient_verdicts = assign_all_verdicts(
    bootstrap_results  = ci_results,
    thresholds         = lenient_thresholds,
    ci_width_threshold = 0.25,
)

print('Lenient threshold verdicts (threshold = 0.20, CI width = 0.25):')
print(f'{"Metric":<30} {"Point":>7}  Verdict')
print('-' * 60)
for key, v in lenient_verdicts.items():
    print(f"{v['label']:<30} {v['point']:>7.4f}  {v['verdict']}")

In [ ]:
# Compare default, strict, and lenient side by side
default_verdicts = assign_all_verdicts(ci_results)

print(f'{"Metric":<10}  {"Default":^30}  {"Strict":^30}  {"Lenient":^30}')
print('-' * 105)
for key in ['dpd', 'eod', 'eop', 'di']:
    dv = default_verdicts[key]['verdict']
    sv = strict_verdicts[key]['verdict']
    lv = lenient_verdicts[key]['verdict']
    print(f'{key.upper():<10}  {dv:<30}  {sv:<30}  {lv:<30}')

In [ ]:
# Custom thresholds also work with assign_verdict for a single metric
single = assign_verdict(
    metric_key         = 'dpd',
    point              = ci_results['dpd']['point'],
    ci_lo              = ci_results['dpd']['ci_lo'],
    ci_hi              = ci_results['dpd']['ci_hi'],
    thresholds         = strict_thresholds,
    ci_width_threshold = 0.10,
)
print(f"DPD under strict thresholds: {single['verdict']}")

---
## Section 10: Saving Results

You can save audit results to CSV for further analysis, and CI plots to PNG for reports or thesis figures.

In [ ]:
# Save the verdict table to CSV
df_results = report.to_dataframe()
df_results.to_csv('compas_lr_audit_results.csv', index=False)
print('Audit results saved to compas_lr_audit_results.csv')
print()
print(df_results.to_string(index=False))

In [ ]:
# Build a results table manually from the verdicts dict
# Useful when you ran the stages individually rather than through AuditReport
rows = []
for key, v in verdicts.items():
    rows.append({
        'metric'    : key,
        'label'     : v['label'],
        'point'     : v['point'],
        'ci_lo'     : v['ci_lo'],
        'ci_hi'     : v['ci_hi'],
        'ci_width'  : v['width'],
        'pass_fail' : v['pass_fail'],
        'confidence': v['confidence'],
        'verdict'   : v['verdict'],
    })

results_df = pd.DataFrame(rows)
results_df.to_csv('compas_lr_manual_results.csv', index=False)
print('Manual results saved to compas_lr_manual_results.csv')

In [ ]:
# Save CI plot and verdict grid as images
fig_ci   = plot_ci_intervals(verdicts, title='COMPAS LR — CI Plot')
fig_ci.savefig('compas_lr_ci_intervals.png', dpi=150, bbox_inches='tight')
print('CI plot saved to compas_lr_ci_intervals.png')

fig_grid = plot_verdict_summary(
    {'Logistic Regression': verdicts, 'Random Forest': verdicts_rf},
    title='COMPAS — Model Comparison Grid'
)
fig_grid.savefig('compas_model_comparison.png', dpi=150, bbox_inches='tight')
print('Verdict grid saved to compas_model_comparison.png')

plt.show()

---
## Quick Reference

### Import summary

```python
from uabaf import AuditReport                          # full pipeline in one call
from uabaf.profiler  import profile_dataset            # Stage 1 only
from uabaf.metrics   import fairness_metrics           # point estimates only
from uabaf.metrics   import demographic_parity_diff    # individual metric
from uabaf.metrics   import equalized_odds_diff        # individual metric
from uabaf.metrics   import equal_opportunity_diff     # individual metric
from uabaf.metrics   import disparate_impact_ratio     # individual metric
from uabaf.bootstrap import bca_ci                     # single metric CI
from uabaf.bootstrap import bca_ci_all_metrics         # all metrics CI
from uabaf.verdict   import assign_verdict             # single metric verdict
from uabaf.verdict   import assign_all_verdicts        # all metric verdicts
from uabaf.verdict   import THRESHOLDS                 # default threshold values
from uabaf.verdict   import CI_WIDTH_THRESHOLD         # default 0.15
from uabaf.visualise import plot_ci_intervals          # CI bar chart
from uabaf.visualise import plot_verdict_summary       # multi-model grid
```

### Fairness thresholds

| Metric | Fair if | Default threshold |
|--------|---------|------------------|
| DPD | \|value\| <= threshold | 0.10 |
| EOD | value <= threshold | 0.10 |
| EOP | value <= threshold | 0.10 |
| DI  | lo <= value <= hi | 0.80 to 1.20 |

CI width threshold: **0.15** (values below this are high confidence)

### Verdict logic

```
Point estimate passes  +  CI width <= 0.15  =  PASS with High Confidence
Point estimate passes  +  CI width >  0.15  =  PASS with Low Confidence
Point estimate fails   +  CI width <= 0.15  =  FAIL with High Confidence
Point estimate fails   +  CI width >  0.15  =  FAIL with Low Confidence
```

---
**Package:** `pip install uabaf`  
**GitHub:** https://github.com/YourGitHubUsername/uabaf  
**PyPI:** https://pypi.org/project/uabaf